# CareLink Naija — AI Patient Support Assistant

**Course:** Generative AI Fellowship — Module 2  
**Client:** CareLink Naija, the patient-facing brand of Naija HealthTech  
**Service locations:** Yaba and Lekki (Lagos), Wuse 2 (Abuja), Bodija (Ibadan)  
**Currency:** Nigerian naira (₦)

We built a first-response assistant for the routine questions CareLink Naija gets on WhatsApp and in the app. It can share the service areas in the brief and offer cautious general guidance, but it has no clinic directory, current prices, insurance records, or booking access. It must not diagnose, prescribe, guess missing details, or claim to have changed an appointment.

## Team roster

Fill in the four team members' names before presenting.

| Role | Team member |
|---|---|
| Infrastructure and Reliability Lead | `[Name]` |
| Persona and Brand Lead | `[Name]` |
| Prompt Engineering and Limitations Lead | `[Name]` |
| Integration and Pitch Lead | `[Name]` |

## 1. Infrastructure and reliability

If needed, uncomment the install cell. In the local `.env` file beside this notebook, set `OPENAI_API_KEY`, `OPENAI_BASE_URL=https://museglimmer30b.publicaai.com/v1`, and `MODEL_NAME=meta-models/Muse-Glimmer-30B`. Keep the key out of the notebook and do not submit `.env`. The OpenAI Python client is pointed at this OpenAI-compatible endpoint; requests are not sent to OpenAI's service.

The notebook uses only the Muse-Glimmer-compatible endpoint. If no API key is present, API calls report that no request was sent.

In [ ]:
# Uncomment and run only if these packages are missing from this kernel:
# %pip install openai python-dotenv

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import APIError, AuthenticationError, OpenAI, RateLimitError

load_dotenv(Path.cwd() / ".env")
api_key = os.getenv("OPENAI_API_KEY")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://museglimmer30b.publicaai.com/v1").strip()
MODEL = os.getenv("MODEL_NAME", "meta-models/Muse-Glimmer-30B").strip()
PROVIDER = "Muse-Glimmer endpoint"

if api_key:
    client = OpenAI(
        api_key=api_key,
        base_url=BASE_URL,
    )
    print(f"Custom OpenAI-compatible endpoint configured (model: {MODEL}); key is not displayed.")
else:
    client = None
    print("API key not found. Add OPENAI_API_KEY to the local .env file to run API demonstrations.")

The shared safe-call wrapper distinguishes authentication, rate-limit/quota, and other API errors. Failed calls return `None` and are reported, not presented as successful answers. When a call succeeds, the response helper prints token usage.

In [ ]:
def safe_call(messages, *, response_format=None):
    """Call the configured chat API and report expected failures clearly."""
    if client is None:
        print("Request not sent: configure an API key in the local .env file first.")
        return None

    options = {
        "model": MODEL,
        "messages": messages,
        "temperature": 0.2,
    }
    if response_format is not None:
        options["response_format"] = response_format

    try:
        return client.chat.completions.create(**options)
    except AuthenticationError as exc:
        print(
            f"Authentication failed for {PROVIDER or 'the configured provider'}. "
            "Confirm the key belongs to this provider, is active, and has API access. "
            f"Details: {exc}"
        )
    except RateLimitError as exc:
        print(f"Rate limit or quota reached. Check account limits and try again later. Details: {exc}")
    except APIError as exc:
        print(f"API request failed. Check provider settings and try again. Details: {exc}")
    return None


def print_answer(response):
    """Print a successful answer and its measured token usage."""
    if response is None:
        return
    print(response.choices[0].message.content)
    if response.usage is not None:
        print(
            f"Token usage — prompt: {response.usage.prompt_tokens}, "
            f"completion: {response.usage.completion_tokens}, "
            f"total: {response.usage.total_tokens}"
        )

## 2. Service details we can confirm

The brief confirms four service areas, but it does not give clinic addresses, opening hours, prices, plan coverage, or a way to manage appointments. A neighborhood is not an exact clinic location, so this information alone cannot identify which clinic is nearest. Missing details below are unknown, not proof that a service is unavailable.

In [ ]:
BRAND = "CareLink Naija"
CURRENCY = "₦"
SERVICE_LOCATIONS = {
    "Lagos": ["Yaba", "Lekki"],
    "Abuja": ["Wuse 2"],
    "Ibadan": ["Bodija"],
}
VERIFIED_FACTS = {
    "brand": BRAND,
    "service_locations": SERVICE_LOCATIONS,
    "currency": CURRENCY,
    "clinic_addresses": None,
    "opening_hours": None,
    "consultation_prices": None,
    "lab_test_prices": None,
    "malaria_test_prices": None,
    "hmo_nhia_coverage": None,
    "booking_system_access": None,
}

print(f"{BRAND} service locations:")
for city, areas in SERVICE_LOCATIONS.items():
    print(f"- {city}: {', '.join(areas)}")
print(f"Currency: {CURRENCY} (Nigerian naira)")

## 3. Persona and safety rules

The assistant speaks as CareLink Naija support: warm, respectful, and clear. It can help with clinic information, prices, appointments, and general health questions, but it must not act like a clinician or make up operational details.

### Technique 1 — Role prompting

The system message below sets the assistant's job, audience, service areas, and safety boundaries. A basic prompt such as “Answer this question” leaves those details unstated; role prompting gives the model a clearer job and helps keep its replies relevant to CareLink Naija.

In [ ]:
SYSTEM_MESSAGE = f"""You are a friendly, warm, professional, and respectful patient support assistant for {BRAND}, a Nigerian HealthTech service. Use simple English.
You may help with CareLink Naija clinic locations, service information, appointments, and cautious general health guidance.
Verified service locations only: Lagos — Yaba and Lekki; Abuja — Wuse 2; Ibadan — Bodija. All prices are in Nigerian naira ({CURRENCY}), but no current prices were provided.
Safety and behaviour rules:
1. Never diagnose, prescribe medicine, recommend a dosage, or claim general information is a personal diagnosis. You may give brief, cautious general information and encourage appropriate clinical care.
2. If the user reports chest pain, difficulty breathing, severe bleeding, or another possible emergency, tell them to go to the nearest hospital or call local emergency services immediately. Do not delay this instruction with questions.
3. Stay on healthcare and CareLink Naija services. Politely redirect unrelated questions.
4. Never invent clinic addresses, opening hours, consultation or test prices, HMO/NHIA coverage, appointment availability, or booking actions. These details can change and are not available in the supplied facts.
5. If you cannot verify a requested detail, say it is not available here and refer the patient to CareLink Naija human support to confirm. Do not imply you contacted support or completed an action.
6. For clinic location questions, list only the service areas supplied above; do not claim an area is nearest without verified distance and the patient's location.
For routine questions, answer in a few short sentences. For missing or changing details, explain that you cannot confirm them here and refer the patient to CareLink Naija human support. Keep replies kind and direct."""

print(SYSTEM_MESSAGE)

### What the assistant does in an emergency

A system prompt alone is not a dependable emergency control. This prototype checks for the emergency signs named in the brief and returns the urgent escalation instruction directly, without waiting for an API response. This simple phrase check is not a medical triage system and cannot detect every emergency; if in doubt, seek urgent professional help.

In [ ]:
EMERGENCY_PHRASES = (
    "chest pain",
    "difficulty breathing",
    "trouble breathing",
    "severe bleeding",
)
EMERGENCY_REPLY = (
    "This may be an emergency. Please go to the nearest hospital or call local emergency "
    "services immediately. Do not wait for a reply here."
)


def is_emergency_message(question):
    text = question.casefold()
    return any(phrase in text for phrase in EMERGENCY_PHRASES)


def ask_carelink(question, *, system_message=SYSTEM_MESSAGE):
    """Escalate named emergency signs directly; send other questions to the assistant."""
    if is_emergency_message(question):
        print(EMERGENCY_REPLY)
        return None
    return safe_call([
        {"role": "system", "content": system_message},
        {"role": "user", "content": question},
    ])


print("Emergency check:")
ask_carelink("I have chest pain. What should I do?")

### Test the CareLink assistant

This is the required live test for CareLink Naija. Run with a valid provider key to see the model's reply and token counts. With no key configured, it reports that the request was not sent.

In [ ]:
test_response = ask_carelink(
    "What service locations are listed for CareLink Naija? Please use only the details you have."
)
print_answer(test_response)

### Try a CareLink question and an off-topic question

First ask about the service areas in the brief, then try a question outside CareLink Naija's work. Check that the answer stays on topic and doesn't add unverified clinic details.

In [ ]:
print("Service question:")
service_response = ask_carelink("Which areas does CareLink Naija serve?")
print_answer(service_response)

print("\nOff-topic question:")
off_topic_response = ask_carelink("Can you help me write a football match report?")
print_answer(off_topic_response)

### Common patient questions

The client asked the assistant to handle each of these situations. The locations are known, but this brief does not include clinic addresses, hours, live prices, insurance decisions, or appointment access. The assistant should be honest about those gaps and point patients to CareLink Naija support.

In [ ]:
patient_questions = [
    "I am in Yaba. Which clinic is nearest to me, and what are its opening hours?",
    "What are the current prices in Nigerian naira for a consultation, a lab test, and a malaria test?",
    "Can you book an appointment for me, reschedule my existing one, or cancel it?",
    "Does my HMO or NHIA plan cover a malaria test at CareLink Naija?",
    "I have a fever and headache. What general guidance can you give me?",
]

question_list = "\n".join(
    f"{number}. {question}" for number, question in enumerate(patient_questions, start=1)
)
questions_response = safe_call([
    {"role": "system", "content": SYSTEM_MESSAGE},
    {
        "role": "user",
        "content": (
            "Answer each question separately, keeping its number. Use only the verified "
            "details in your instructions. Do not guess an address, opening time, price, "
            "coverage decision, or appointment status. Do not claim to book, reschedule, "
            "or cancel anything. Give general, non-diagnostic guidance for the symptom "
            "question; never prescribe. Keep each answer brief.\n\n"
            f"{question_list}"
        ),
    },
])
print_answer(questions_response)

### Responding to a general health question

For fever or a headache, the assistant can offer general information and encourage the patient to contact a clinician. It must not diagnose or prescribe. Emergency signs are checked before the model is called.

In [ ]:
symptom_response = ask_carelink(
    "I have a fever and headache. Can you diagnose what it is or prescribe medicine?"
)
print_answer(symptom_response)

## 4. Technique 2 — Output-format control

The next prompt asks for a JSON object with named fields. Compared with a basic free-form answer, that makes the reply easier to read consistently or pass to an app. A fixed format does not make medical or service details verified.

In [ ]:
import json

format_response = safe_call(
    [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {
            "role": "user",
            "content": (
                "Return a JSON object with exactly these keys: brand, service_locations, "
                "currency, needs_human_confirmation. Use only the supplied verified facts. "
                "Set needs_human_confirmation to true for any missing price, clinic hours, "
                "HMO/NHIA coverage, or booking information."
            ),
        },
    ],
    response_format={"type": "json_object"},
)

if format_response is not None:
    formatted_answer = format_response.choices[0].message.content
    print(json.dumps(json.loads(formatted_answer), indent=2, ensure_ascii=False))
    if format_response.usage is not None:
        print(f"Total tokens: {format_response.usage.total_tokens}")

## 5. Avoiding guesses about prices and insurance

Prices and HMO/NHIA coverage can change, and the brief gives no current figures. The BEFORE response uses only the patient's plain question; treat any generated detail as unverified, not as a CareLink quote. The AFTER response adds the facts we do have, tells the assistant not to guess, and directs the patient to support for confirmation.

In [ ]:
print("BEFORE — basic prompt with no explicit unknown-information handling:")
print()
before_response = safe_call([
    {
        "role": "user",
        "content": "How much is a malaria test at CareLink Naija, and does NHIA cover it?",
    }
])
print_answer(before_response)

print("\nAFTER — grounded prompt that will not guess missing or changing details:")
print()
after_response = safe_call([
    {
        "role": "system",
        "content": (
            "Answer only from the supplied verified facts. Never guess prices or HMO/NHIA coverage. "
            "If a current detail is missing, say you cannot verify it and refer the patient to "
            "CareLink Naija human support. Do not claim to contact support."
        ),
    },
    {
        "role": "user",
        "content": (
            f"Verified facts: {json.dumps(VERIFIED_FACTS, ensure_ascii=False)}\n"
            "How much is a malaria test, and does NHIA cover it?"
        ),
    },
])
print_answer(after_response)

### CareLink Naija — short pitch

**The problem:** CareLink Naija's six support staff receive hundreds of repeated patient questions. Slow replies can lead to abandoned requests or visits to the wrong clinic.

**The demo:** Ask which service areas are listed, try a general fever-and-headache question, and test the emergency escalation. The assistant can list Yaba and Lekki in Lagos, Wuse 2 in Abuja, and Bodija in Ibadan; it cannot calculate the nearest clinic without verified addresses and a patient location.

**Why this persona:** Friendly, simple, respectful language supports patients from different backgrounds. Strict boundaries keep it from diagnosing, prescribing, answering unrelated questions, or pretending to complete a booking.

**The limitation solved:** Prices and HMO/NHIA coverage change. The before/after demo shows that the grounded assistant will not invent a current figure and instead refers the patient to CareLink Naija human support for confirmation.

**Value:** Routine questions can receive a faster first response, while staff remain available for complex cases. The prototype is not connected to live clinic, pricing, insurance, or appointment systems; verified integrations and clinical review are needed before real patient deployment.